In [1]:
from pathlib import Path

from pydantic import BaseModel, Field


class Symbol(BaseModel):
    name: str
    kind: str
    line_start: int
    line_end: int


class FileRecord(BaseModel):
    path: Path
    language: str
    content: str
    symbols: list[Symbol] = Field(default_factory=list)
    imports: list[str] = Field(default_factory=list)


class CodeChunk(BaseModel):
    file_path: Path
    start_line: int
    end_line: int
    content: str
    symbols: list[str] = Field(default_factory=list)


class RankedChunk(BaseModel):
    chunk: CodeChunk
    score: float
    reasons: list[str] = Field(default_factory=list)


class QueryRequest(BaseModel):
    query: str
    top_k: int | None = Field(default=None, ge=1)


class ChatRequest(BaseModel):
    message: str
    top_k: int | None = Field(default=None, ge=1)


class IndexRequest(BaseModel):
    root_path: Path


class QueryResponse(BaseModel):
    query: str
    indexed_root: Path | None
    total_files: int
    total_chunks: int
    context_pack: str
    ranked_chunks: list[RankedChunk]


class ChatResponse(BaseModel):
    message: str
    indexed_root: Path | None
    model_provider: str
    model_name: str | None
    context_pack: str
    answer: str


In [2]:
class ContextBuilder:
    def build(self, query: str, chunks: list[RankedChunk]) -> str:
        lines = [
            f"User Request:\n{query}",
            "",
            "Relevant Context:",
        ]

        for index, item in enumerate(chunks, start=1):
            lines.extend(
                [
                    f"{index}. File: {item.chunk.file_path}",
                    f"   Lines: {item.chunk.start_line}-{item.chunk.end_line}",
                    f"   Score: {item.score}",
                    f"   Reasons: {', '.join(item.reasons) or 'retrieval match'}",
                    "   Snippet:",
                    self._indent(item.chunk.content.strip() or "<empty>"),
                    "",
                ]
            )

        return "\n".join(lines).rstrip()

    def _indent(self, text: str) -> str:
        return "\n".join(f"   {line}" for line in text.splitlines())

In [3]:
import os
import tomllib
from pathlib import Path
from typing import Literal

from pydantic import BaseModel, Field


LlmProvider = Literal["openai_compatible", "openrouter", "gemini", "none"]
EmbeddingProvider = Literal["openai_compatible", "ollama", "sentence_transformers", "none"]


class LlmConfig(BaseModel):
    provider: LlmProvider = "none"
    model: str | None = None
    base_url: str | None = None
    api_key: str | None = None
    api_key_env: str | None = None
    timeout_seconds: float = Field(default=60.0, gt=0)
    temperature: float = Field(default=0.1, ge=0, le=2)

    def resolved_api_key(self) -> str | None:
        if self.api_key:
            return self.api_key
        if self.api_key_env:
            return os.getenv(self.api_key_env)
        return None


class EmbeddingConfig(BaseModel):
    provider: EmbeddingProvider = "none"
    model: str | None = None
    base_url: str | None = None
    api_key: str | None = None
    api_key_env: str | None = None
    timeout_seconds: float = Field(default=60.0, gt=0)

    def resolved_api_key(self) -> str | None:
        if self.api_key:
            return self.api_key
        if self.api_key_env:
            return os.getenv(self.api_key_env)
        return None


class EngineConfig(BaseModel):
    max_files: int = Field(default=5000, ge=1)
    max_file_size_bytes: int = Field(default=2_560_000, ge=0)
    default_window: int = Field(default=20, ge=1)
    top_k_chunks: int = Field(default=8, ge=1)
    top_k_files: int = Field(default=5, ge=1)
    symbol_context_padding: int = 0  # Default: precise extraction
    include_extensions: tuple[str, ...] = (
        ".py",
        ".ts",
        ".tsx",
        ".js",
        ".jsx",
        ".java",
        ".go",
        ".rs",
        ".json",
        ".md",
        ".yaml",
        ".yml",
        ".php",  # ✅ Added
    )
    ignore_dirs: tuple[str, ...] = (
        "site-packages",
        ".git",
        ".ipynb_checkpoints",
        ".venv",
        "venv",
        "node_modules",
        "__pycache__",
        "dist",
        "build",
    )
    workspace_root: Path | None = None


class AppConfig(BaseModel):
    engine: EngineConfig = Field(default_factory=EngineConfig)
    llm: LlmConfig = Field(default_factory=LlmConfig)
    embeddings: EmbeddingConfig = Field(default_factory=EmbeddingConfig)


def load_dotenv(path: Path) -> None:
    if not path.exists():
        return

    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip("'\"")
        if key and key not in os.environ:
            os.environ[key] = value


def load_app_config(path: Path | None = None) -> AppConfig:
    config_path = path or Path("config.toml")
    load_dotenv(config_path.with_name(".env"))
    if not config_path.exists():
        return AppConfig()

    with config_path.open("rb") as handle:
        data = tomllib.load(handle)

    return AppConfig.model_validate(data)

In [4]:
from collections import defaultdict
from pathlib import Path

class DependencyGraph:
    def __init__(self) -> None:
        self.forward: dict[Path, set[str]] = defaultdict(set)

    def build(self, files: list[FileRecord]) -> "DependencyGraph":
        self.forward.clear()
        for record in files:
            self.forward[record.path].update(record.imports)
        return self

    def neighbors(self, path: Path) -> set[str]:
        return self.forward.get(path, set())

In [5]:
from pathlib import Path

EXTENSION_TO_LANGUAGE = {
    ".py": "python",
    ".ts": "typescript",
    ".tsx": "tsx",
    ".js": "javascript",
    ".jsx": "jsx",
    ".java": "java",
    ".go": "go",
    ".rs": "rust",
    ".json": "json",
    ".yaml": "yaml",
    ".yml": "yaml",
    ".php": "php",
}


def detect_language(path: Path) -> str:
    return EXTENSION_TO_LANGUAGE.get(path.suffix.lower(), "text")

In [6]:
from __future__ import annotations

from typing import Dict, List, Tuple
from tree_sitter import Parser, Language
import re

# Official Tree-sitter bindings
import tree_sitter_python as tspython
import tree_sitter_javascript as tsjavascript
import tree_sitter_typescript as tstypescript
import tree_sitter_java as tsjava
import tree_sitter_go as tsgo
import tree_sitter_rust as tsrust
import tree_sitter_json as tsjson
import tree_sitter_yaml as tsyaml
import tree_sitter_php as tsphp

LANGUAGE_MAP: Dict[str, Language] = {
    "python": Language(tspython.language()),
    "javascript": Language(tsjavascript.language()),
    "typescript": Language(tstypescript.language_typescript()),
    "tsx": Language(tstypescript.language_tsx()),
    "jsx": Language(tsjavascript.language()),
    "java": Language(tsjava.language()),
    "go": Language(tsgo.language()),
    "rust": Language(tsrust.language()),
    "json": Language(tsjson.language()),
    "yaml": Language(tsyaml.language()),
    "php": Language(tsphp.language_php()),
}


class TreeSitterParser:
    """Centralized multi-language parser using official Tree-sitter bindings."""

    def __init__(self) -> None:
        self._parsers: Dict[str, Parser] = {}

    def _get_parser(self, language: str) -> Parser | None:
        ts_language = LANGUAGE_MAP.get(language)
        if ts_language is None:
            return None
    
        if language not in self._parsers:
            parser = Parser()
            parser.language = ts_language  # Correct for Tree-sitter ≥ 0.22
            self._parsers[language] = parser
    
        return self._parsers[language]

    def parse(self, content: str, language: str) -> Tuple[List[Symbol], List[str]]:
        parser = self._get_parser(language)
        if not parser:
            return [], []

        tree = parser.parse(content.encode("utf-8"))
        root = tree.root_node

        symbols: List[Symbol] = []
        imports: List[str] = []

        def text(node):
            return content[node.start_byte:node.end_byte]

        # def add_symbol(node, name_node, kind):
        #     if not name_node:
        #         return
        
        #     name = text(name_node).strip()
        
        #     # Filter invalid symbols
        #     if not re.match(r"^[A-Za-z_][A-Za-z0-9_]*$", name):
        #         return
        #     if len(name) <= 1:
        #         return
        #     if name.lower() in {"i", "e", "a", "u", "v", "g", "n", "r", "t", "o", "l"}:
        #         return
        
        #     symbols.append(
        #         Symbol(
        #             name=name,
        #             kind=kind,
        #             line_start=node.start_point[0] + 1,
        #             line_end=node.end_point[0] + 1,
        #         )
        #     )

        def add_symbol(node, name_node, kind):
            if not name_node:
                return
        
            symbol_name = text(name_node)
        
            start_line = node.start_point[0] + 1
            end_line = node.end_point[0] + 1
            start_byte = node.start_byte
            end_byte = node.end_byte
        
            symbols.append(
                Symbol(
                    name=symbol_name,
                    kind=kind,
                    line_start=start_line,
                    line_end=end_line,
                    content=content[start_byte:end_byte],  # Exact AST slice
                )
            )

        def walk(node):
            node_type = node.type

            # Python
            if language == "python":
                if node_type == "function_definition":
                    add_symbol(node, node.child_by_field_name("name"), "function")
                elif node_type == "class_definition":
                    add_symbol(node, node.child_by_field_name("name"), "class")
                elif node_type in {"import_statement", "import_from_statement"}:
                    imports.append(text(node))

            # JavaScript / TypeScript / TSX / JSX
            elif language in {"javascript", "typescript", "tsx", "jsx"}:
                if node_type == "function_declaration":
                    add_symbol(node, node.child_by_field_name("name"), "function")
                elif node_type == "class_declaration":
                    add_symbol(node, node.child_by_field_name("name"), "class")
                elif node_type == "method_definition":
                    add_symbol(node, node.child_by_field_name("name"), "method")
                elif node_type == "import_statement":
                    imports.append(text(node))
                elif node_type == "variable_declarator":
                    name_node = node.child_by_field_name("name")
                    value_node = node.child_by_field_name("value")
                    if value_node and value_node.type in {
                        "arrow_function",
                        "function",
                        "function_expression",
                    }:
                        add_symbol(node, name_node, "function")

            # Java
            elif language == "java":
                if node_type == "class_declaration":
                    add_symbol(node, node.child_by_field_name("name"), "class")
                elif node_type == "method_declaration":
                    add_symbol(node, node.child_by_field_name("name"), "method")
                elif node_type == "import_declaration":
                    imports.append(text(node))

            # Go
            elif language == "go":
                if node_type in {"function_declaration", "method_declaration"}:
                    add_symbol(node, node.child_by_field_name("name"), "function")
                elif node_type == "import_declaration":
                    imports.append(text(node))

            # Rust
            elif language == "rust":
                if node_type == "function_item":
                    add_symbol(node, node.child_by_field_name("name"), "function")
                elif node_type == "struct_item":
                    add_symbol(node, node.child_by_field_name("name"), "struct")
                elif node_type == "use_declaration":
                    imports.append(text(node))

            # ---------------- PHP ----------------
            elif language == "php":
                if node_type == "class_declaration":
                    add_symbol(node, node.child_by_field_name("name"), "class")
            
                elif node_type == "method_declaration":
                    add_symbol(node, node.child_by_field_name("name"), "method")
            
                elif node_type == "function_definition":
                    add_symbol(node, node.child_by_field_name("name"), "function")
            
                elif node_type == "namespace_definition":
                    imports.append(text(node))
            
                elif node_type == "namespace_use_declaration":
                    imports.append(text(node))

            for child in node.children:
                walk(child)

        walk(root)
        return symbols, imports

In [7]:
pwd

'/home/butcher/MyCodeIDE/src/context_engine'

In [8]:
from pathlib import Path

parser = TreeSitterParser()

file_path = Path(r"/home/butcher/MyCodeIDE/src/context_engine/desktop.py")
content = file_path.read_text(encoding="utf-8")

symbols, imports = parser.parse(content, "python")

print("Detected Symbols:")
for sym in symbols[:10]:
    print(sym)

print("\nTotal Symbols:", len(symbols))

Detected Symbols:
name='format_chunk_title' kind='function' line_start=14 line_end=20
name='DesktopApp' kind='class' line_start=23 line_end=495
name='__init__' kind='function' line_start=24 line_end=45
name='_build_ui' kind='function' line_start=47 line_end=291
name='_browse_folder' kind='function' line_start=293 line_end=296
name='_set_busy' kind='function' line_start=298 line_end=300
name='_clear_busy' kind='function' line_start=302 line_end=303
name='_start_index' kind='function' line_start=305 line_end=321
name='_run_index' kind='function' line_start=323 line_end=328
name='_start_run' kind='function' line_start=330 line_end=349

Total Symbols: 23


In [9]:
from pathlib import Path
from typing import Callable, Iterable

class CodebaseIndexer:
    def __init__(self, config: EngineConfig) -> None:
        self.config = config
        self.parser = TreeSitterParser()

    def index(
        self,
        root: Path,
        progress_callback: Callable[[int, int, Path], None] | None = None,
    ) -> tuple[list[FileRecord], list[CodeChunk]]:
        files: list[FileRecord] = []
        chunks: list[CodeChunk] = []
        candidates = list(self._iter_files(root))
        total = len(candidates)

        for index, path in enumerate(candidates, start=1):
            content = path.read_text(encoding="utf-8", errors="ignore")
            record = self._build_file_record(path, content)
            files.append(record)
            chunks.extend(self._chunk_file(record))
            if progress_callback:
                progress_callback(index, total, path)

        return files, chunks

    def index_uploaded(
        self,
        virtual_root: Path,
        uploaded_files: Iterable[tuple[Path, str]],
        progress_callback: Callable[[int, int, Path], None] | None = None,
    ) -> tuple[list[FileRecord], list[CodeChunk]]:
        files: list[FileRecord] = []
        chunks: list[CodeChunk] = []
        accepted_files: list[tuple[Path, str]] = []

        for relative_path, content in uploaded_files:
            if len(accepted_files) >= self.config.max_files:
                break

            normalized = Path(str(relative_path).replace("\\", "/"))
            if normalized.suffix.lower() not in self.config.include_extensions:
                continue
            if any(part in self.config.ignore_dirs for part in normalized.parts):
                continue
            if len(content.encode("utf-8", errors="ignore")) > self.config.max_file_size_bytes:
                continue

            accepted_files.append((normalized, content))

        total = len(accepted_files)
        for index, (normalized, content) in enumerate(accepted_files, start=1):
            record = self._build_file_record(virtual_root / normalized, content)
            files.append(record)
            chunks.extend(self._chunk_file(record))
            if progress_callback:
                progress_callback(index, total, virtual_root / normalized)

        return files, chunks

    def _iter_files(self, root: Path):
        count = 0
        for path in root.rglob("*"):
            if not path.is_file():
                continue
            if path.suffix.lower() not in self.config.include_extensions:
                continue
            if any(part in self.config.ignore_dirs for part in path.parts):
                continue
            try:
                if path.stat().st_size > self.config.max_file_size_bytes:
                    continue
            except OSError:
                continue
            yield path
            count += 1
            if count >= self.config.max_files:
                break

    def _build_file_record(self, path: Path, content: str) -> FileRecord:
        language = detect_language(path)
        symbols, imports = self.parser.parse(content, language)
    
        return FileRecord(
            path=path,
            language=language,
            content=content,
            symbols=symbols,
            imports=imports,
        )

    def _chunk_file(self, record: FileRecord) -> list[CodeChunk]:
        """Generate precise symbol-based chunks with optional context padding."""
        chunks: list[CodeChunk] = []
    
        # Preferred: Symbol-based chunking
        if record.symbols:
            lines = record.content.splitlines()
            padding = getattr(self.config, "symbol_context_padding", 0)
    
            for symbol in record.symbols:
                # Case 1: Exact AST-based extraction (no padding)
                if padding == 0 and getattr(symbol, "content", None):
                    content = symbol.content.strip()
                    start_line = symbol.line_start
                    end_line = symbol.line_end
    
                # Case 2: Context-aware extraction with padding
                else:
                    start_line = max(symbol.line_start - padding, 1)
                    end_line = min(symbol.line_end + padding, len(lines))
                    content = "\n".join(lines[start_line - 1:end_line]).strip()
    
                if not content:
                    continue
    
                chunks.append(
                    CodeChunk(
                        file_path=record.path,
                        start_line=start_line,
                        end_line=end_line,
                        content=content,
                        symbols=[symbol.name],
                    )
                )
    
            return chunks
    
        # Fallback: Window-based chunking for files without symbols
        lines = record.content.splitlines()
        if not lines:
            return []
    
        window = max(self.config.default_window, 1)
    
        for start in range(0, len(lines), window):
            end = min(start + window, len(lines))
            chunks.append(
                CodeChunk(
                    file_path=record.path,
                    start_line=start + 1,
                    end_line=end,
                    content="\n".join(lines[start:end]),
                    symbols=[],
                )
            )
    
        return chunks


In [10]:
import math
from typing import Protocol

import httpx


class EmbeddingProvider(Protocol):
    def embed_texts(self, texts: list[str]) -> list[list[float]]:
        ...


class LlmProvider(Protocol):
    def chat(self, messages: list[dict[str, str]], temperature: float) -> str:
        ...


class NullEmbeddingProvider:
    def embed_texts(self, texts: list[str]) -> list[list[float]]:
        return [[] for _ in texts]


class NullLlmProvider:
    def chat(self, messages: list[dict[str, str]], temperature: float) -> str:
        raise RuntimeError("No LLM provider is configured.")


def raise_for_status_with_detail(response: httpx.Response, provider_name: str) -> None:
    try:
        response.raise_for_status()
    except httpx.HTTPStatusError as exc:
        detail = response.text.strip()
        if response.status_code == 401:
            raise RuntimeError(
                f"{provider_name} returned 401 Unauthorized. Check the configured API key and .env loading."
            ) from exc
        if detail:
            raise RuntimeError(f"{provider_name} request failed: {response.status_code} {detail}") from exc
        raise


class OpenAICompatibleEmbeddingProvider:
    def __init__(self, config: EmbeddingConfig) -> None:
        if not config.base_url or not config.model:
            raise ValueError("OpenAI-compatible embeddings require base_url and model.")
        self.base_url = config.base_url.rstrip("/")
        self.model = config.model
        self.api_key = config.resolved_api_key()
        self.timeout = config.timeout_seconds

    def embed_texts(self, texts: list[str]) -> list[list[float]]:
        headers = {"Content-Type": "application/json"}
        if self.api_key:
            headers["Authorization"] = f"Bearer {self.api_key}"

        with httpx.Client(timeout=self.timeout) as client:
            response = client.post(
                f"{self.base_url}/embeddings",
                headers=headers,
                json={"model": self.model, "input": texts},
            )
            raise_for_status_with_detail(response, "Embedding provider")
            data = response.json()
        return [item["embedding"] for item in data["data"]]


class OllamaEmbeddingProvider:
    def __init__(self, config: EmbeddingConfig) -> None:
        self.base_url = (config.base_url or "http://localhost:11434").rstrip("/")
        self.model = config.model or "nomic-embed-text"
        self.timeout = config.timeout_seconds

    def embed_texts(self, texts: list[str]) -> list[list[float]]:
        vectors: list[list[float]] = []
        with httpx.Client(timeout=self.timeout) as client:
            for text in texts:
                response = client.post(
                    f"{self.base_url}/api/embed",
                    json={"model": self.model, "input": text},
                )
                if response.status_code == 404:
                    response = client.post(
                        f"{self.base_url}/api/embeddings",
                        json={"model": self.model, "prompt": text},
                    )
                raise_for_status_with_detail(response, "Ollama embeddings")
                data = response.json()
                if "embeddings" in data:
                    vectors.append(data["embeddings"][0])
                else:
                    vectors.append(data["embedding"])
        return vectors


class SentenceTransformerEmbeddingProvider:
    def __init__(self, config: EmbeddingConfig) -> None:
        if not config.model:
            raise ValueError("Sentence Transformers embeddings require a model name.")
        try:
            from sentence_transformers import SentenceTransformer
        except ImportError as exc:
            raise RuntimeError(
                "sentence-transformers is not installed. Install with `pip install -e .[embeddings]`."
            ) from exc

        self.model = SentenceTransformer(config.model)

    def embed_texts(self, texts: list[str]) -> list[list[float]]:
        vectors = self.model.encode(texts, normalize_embeddings=True)
        return [vector.tolist() for vector in vectors]


class OpenAICompatibleLlmProvider:
    def __init__(self, config: LlmConfig, extra_headers: dict[str, str] | None = None) -> None:
        if not config.base_url or not config.model:
            raise ValueError("OpenAI-compatible chat requires base_url and model.")
        self.base_url = config.base_url.rstrip("/")
        self.model = config.model
        self.api_key = config.resolved_api_key()
        self.timeout = config.timeout_seconds
        self.extra_headers = extra_headers or {}

    def chat(self, messages: list[dict[str, str]], temperature: float) -> str:
        headers = {"Content-Type": "application/json", **self.extra_headers}
        if self.api_key:
            headers["Authorization"] = f"Bearer {self.api_key}"

        with httpx.Client(timeout=self.timeout) as client:
            response = client.post(
                f"{self.base_url}/chat/completions",
                headers=headers,
                json={
                    "model": self.model,
                    "messages": messages,
                    "temperature": temperature,
                },
            )
            raise_for_status_with_detail(response, "LLM provider")
            data = response.json()
        return data["choices"][0]["message"]["content"]


class GeminiLlmProvider:
    def __init__(self, config: LlmConfig) -> None:
        if not config.model:
            raise ValueError("Gemini chat requires a model.")
        self.model = config.model
        self.api_key = config.resolved_api_key()
        if not self.api_key:
            raise ValueError("Gemini chat requires an API key.")
        self.timeout = config.timeout_seconds
        self.base_url = (config.base_url or "https://generativelanguage.googleapis.com/v1beta").rstrip("/")

    def chat(self, messages: list[dict[str, str]], temperature: float) -> str:
        contents = []
        for message in messages:
            role = "user" if message["role"] == "user" else "model" if message["role"] == "assistant" else "user"
            contents.append({"role": role, "parts": [{"text": message["content"]}]})

        with httpx.Client(timeout=self.timeout) as client:
            response = client.post(
                f"{self.base_url}/models/{self.model}:generateContent",
                params={"key": self.api_key},
                json={
                    "contents": contents,
                    "generationConfig": {"temperature": temperature},
                },
            )
            raise_for_status_with_detail(response, "Gemini")
            data = response.json()

        candidates = data.get("candidates", [])
        if not candidates:
            raise RuntimeError("Gemini returned no candidates.")
        parts = candidates[0].get("content", {}).get("parts", [])
        return "\n".join(part.get("text", "") for part in parts).strip()


def create_embedding_provider(config: EmbeddingConfig) -> EmbeddingProvider:
    if config.provider == "none":
        return NullEmbeddingProvider()
    if config.provider == "openai_compatible":
        return OpenAICompatibleEmbeddingProvider(config)
    if config.provider == "ollama":
        return OllamaEmbeddingProvider(config)
    if config.provider == "sentence_transformers":
        return SentenceTransformerEmbeddingProvider(config)
    raise ValueError(f"Unsupported embedding provider: {config.provider}")


def create_llm_provider(config: LlmConfig) -> LlmProvider:
    if config.provider == "none":
        return NullLlmProvider()
    if config.provider == "openai_compatible":
        return OpenAICompatibleLlmProvider(config)
    if config.provider == "openrouter":
        return OpenAICompatibleLlmProvider(
            config,
            extra_headers={
                "HTTP-Referer": "https://mycodeide.local",
                "X-Title": "MyCodeIDE Context Engine",
            },
        )
    if config.provider == "gemini":
        return GeminiLlmProvider(config)
    raise ValueError(f"Unsupported LLM provider: {config.provider}")


def cosine_similarity(left: list[float], right: list[float]) -> float:
    if not left or not right or len(left) != len(right):
        return 0.0
    numerator = sum(a * b for a, b in zip(left, right))
    left_norm = math.sqrt(sum(a * a for a in left))
    right_norm = math.sqrt(sum(b * b for b in right))
    if left_norm == 0 or right_norm == 0:
        return 0.0
    return numerator / (left_norm * right_norm)


In [11]:
import difflib
import math
import re
from collections import Counter


TOKEN_RE = re.compile(r"[A-Za-z_][A-Za-z0-9_]*")
CAMEL_RE = re.compile(r"[A-Z]+(?=[A-Z][a-z]|\d|_|$)|[A-Z]?[a-z]+|\d+")


def tokenize(text: str) -> list[str]:
    tokens: list[str] = []
    for token in TOKEN_RE.findall(text):
        lowered = token.lower()
        tokens.append(lowered)
        for part in token.split("_"):
            if not part:
                continue
            for camel_part in CAMEL_RE.findall(part):
                normalized = camel_part.lower()
                if normalized and normalized != lowered:
                    tokens.append(normalized)
    return tokens


class HybridRetriever:
    def __init__(
        self,
        files: list[FileRecord],
        chunks: list[CodeChunk],
        graph: DependencyGraph,
        embedding_provider: EmbeddingProvider | None = None,
    ) -> None:
        self.files = files
        self.chunks = chunks
        self.graph = graph
        self.embedding_provider = embedding_provider or NullEmbeddingProvider()
        self.chunk_terms = [tokenize(chunk.content) for chunk in self.chunks]
        self.document_frequency = self._compute_document_frequency()
        self.average_chunk_length = self._compute_average_chunk_length()
        self.chunk_embeddings = self._build_chunk_embeddings()

    def _compute_document_frequency(self) -> Counter[str]:
        counter: Counter[str] = Counter()
        for chunk in self.chunks:
            counter.update(set(tokenize(chunk.content)))
        return counter

    def _compute_average_chunk_length(self) -> float:
        if not self.chunk_terms:
            return 0.0
        return sum(len(terms) for terms in self.chunk_terms) / len(self.chunk_terms)

    def _build_chunk_embeddings(self) -> list[list[float]]:
        if not self.chunks:
            return []
        return self.embedding_provider.embed_texts([chunk.content for chunk in self.chunks])

    def search(self, query: str, top_k: int) -> list[RankedChunk]:
        query_tokens = tokenize(query)
        if not query_tokens:
            return []

        query_embedding = self.embedding_provider.embed_texts([query])[0] if self.chunk_embeddings else []
        ranked: list[RankedChunk] = []
        file_lookup = {record.path: record for record in self.files}

        for index, chunk in enumerate(self.chunks):
            bm25 = self._bm25_score(query_tokens, index)
            fuzzy = self._fuzzy_score(query_tokens, chunk, file_lookup)
            semantic = self._semantic_score(query_embedding, index)
            structural = self._structural_score(query_tokens, chunk, file_lookup)
            dependency = self._dependency_score(query_tokens, chunk, file_lookup)
            score = 0.35 * bm25 + 0.20 * fuzzy + 0.25 * semantic + 0.15 * structural + 0.05 * dependency
            if score <= 0:
                continue

            reasons: list[str] = []
            if bm25:
                reasons.append("bm25 keyword match")
            if fuzzy:
                reasons.append("fuzzy match")
            if semantic:
                reasons.append("semantic similarity")
            if structural:
                reasons.append("symbol/path match")
            if dependency:
                reasons.append("dependency hint")

            ranked.append(RankedChunk(chunk=chunk, score=round(score, 4), reasons=reasons))

        ranked.sort(key=lambda item: item.score, reverse=True)
        return ranked[:top_k]

    def _bm25_score(self, query_tokens: list[str], chunk_index: int) -> float:
        if chunk_index >= len(self.chunk_terms):
            return 0.0
        terms = self.chunk_terms[chunk_index]
        if not terms:
            return 0.0

        counts = Counter(terms)
        total_docs = max(len(self.chunks), 1)
        avg_len = max(self.average_chunk_length, 1.0)
        doc_len = len(terms)
        k1 = 1.5
        b = 0.75
        score = 0.0
        for token in query_tokens:
            df = self.document_frequency.get(token, 0)
            tf = counts[token]
            if tf == 0:
                continue
            idf = math.log(1 + ((total_docs - df + 0.5) / (df + 0.5)))
            numerator = tf * (k1 + 1)
            denominator = tf + k1 * (1 - b + b * (doc_len / avg_len))
            score += idf * (numerator / denominator)
        return score

    def _fuzzy_score(
        self,
        query_tokens: list[str],
        chunk: CodeChunk,
        file_lookup: dict,
    ) -> float:
        candidates = set(tokenize(chunk.content))
        candidates.update(tokenize(str(chunk.file_path)))
        candidates.update(token.lower() for symbol in chunk.symbols for token in tokenize(symbol))
        record = file_lookup.get(chunk.file_path)
        if record:
            candidates.update(tokenize(" ".join(record.imports)))
        if not candidates:
            return 0.0

        score = 0.0
        for token in query_tokens:
            best = 0.0
            for candidate in candidates:
                if token == candidate:
                    best = 1.0
                    break
                ratio = difflib.SequenceMatcher(None, token, candidate).ratio()
                if ratio > best:
                    best = ratio
            if best >= 0.82:
                score += best
        return score / max(len(query_tokens), 1)

    def _semantic_score(self, query_embedding: list[float], chunk_index: int) -> float:
        if not query_embedding or chunk_index >= len(self.chunk_embeddings):
            return 0.0
        return max(cosine_similarity(query_embedding, self.chunk_embeddings[chunk_index]), 0.0)

    def _structural_score(
        self,
        query_tokens: list[str],
        chunk: CodeChunk,
        file_lookup: dict,
    ) -> float:
        score = 0.0
        path_tokens = tokenize(str(chunk.file_path))
        symbol_tokens = [token.lower() for symbol in chunk.symbols for token in tokenize(symbol)]
        for token in query_tokens:
            if token in path_tokens:
                score += 1.0
            if token in symbol_tokens:
                score += 1.5
        return score

    def _dependency_score(
        self,
        query_tokens: list[str],
        chunk: CodeChunk,
        file_lookup: dict,
    ) -> float:
        record = file_lookup.get(chunk.file_path)
        if not record:
            return 0.0
        neighbors = self.graph.neighbors(record.path)
        score = 0.0
        for token in query_tokens:
            if any(token in neighbor.lower() for neighbor in neighbors):
                score += 0.75
        return score


In [12]:
import json
import subprocess
from pathlib import Path
from typing import Dict, List
from urllib.parse import urlparse, unquote


class LSPClient:
    """Minimal yet robust LSP client compatible with Pyright and TS servers."""

    def __init__(self, command: List[str], root_uri: Path):
        self.process = subprocess.Popen(
            command,
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=False,  # Required for JSON-RPC
            bufsize=0,
        )
        self.root_uri = root_uri.resolve().as_uri()
        self._id = 0
        self._opened_documents = set()
        self._initialize()

    def _send(self, payload: Dict):
        if self.process.poll() is not None:
            raise RuntimeError("LSP server terminated unexpectedly.")
    
        message = json.dumps(payload).encode("utf-8")
        header = f"Content-Length: {len(message)}\r\n\r\n".encode("utf-8")
    
        try:
            self.process.stdin.write(header + message)
            self.process.stdin.flush()
        except BrokenPipeError:
            raise RuntimeError("Broken pipe: LSP server is not running.")

    def _receive(self) -> Dict:
        while True:
            line = self.process.stdout.readline()
            if not line:
                raise RuntimeError("LSP server closed unexpectedly.")

            if line.startswith(b"Content-Length"):
                length = int(line.split(b":")[1].strip())
                self.process.stdout.readline()  # Skip blank line
                body = self.process.stdout.read(length)
                return json.loads(body.decode("utf-8"))

    def _request(self, method: str, params: Dict):
        self._id += 1
        request = {
            "jsonrpc": "2.0",
            "id": self._id,
            "method": method,
            "params": params,
        }
        self._send(request)
        return self._receive()

    def _notify(self, method: str, params: Dict):
        notification = {
            "jsonrpc": "2.0",
            "method": method,
            "params": params,
        }
        self._send(notification)

    def _initialize(self):
        self._request(
            "initialize",
            {
                "processId": None,
                "rootUri": self.root_uri,
                "capabilities": {},
                "workspaceFolders": [
                    {"uri": self.root_uri, "name": "workspace"}
                ],
            },
        )
        self._notify("initialized", {})

    def _get_language_id(self, file_path: Path) -> str:
        mapping = {
            ".py": "python",
            ".ts": "typescript",
            ".tsx": "typescriptreact",
            ".js": "javascript",
            ".jsx": "javascriptreact",
            ".json": "json",
            ".php": "php",
        }
        return mapping.get(file_path.suffix.lower(), "plaintext")

    def open_document(self, file_path: Path):
        uri = file_path.resolve().as_uri()
        if uri in self._opened_documents:
            return

        text = file_path.read_text(encoding="utf-8", errors="ignore")

        self._notify(
            "textDocument/didOpen",
            {
                "textDocument": {
                    "uri": uri,
                    "languageId": self._get_language_id(file_path),
                    "version": 1,
                    "text": text,
                }
            },
        )

        self._opened_documents.add(uri)

    def get_definition(self, file_path: Path, line: int, character: int):
        self.open_document(file_path)
        uri = file_path.resolve().as_uri()

        response = self._request(
            "textDocument/definition",
            {
                "textDocument": {"uri": uri},
                "position": {"line": line - 1, "character": character},
            },
        )
        return response.get("result")

    def shutdown(self):
        try:
            self._request("shutdown", {})
            self._notify("exit", {})
        finally:
            self.process.terminate()

In [13]:
import shutil
import sys
from pathlib import Path
from typing import Dict, List, Optional


class LSPManager:
    """Manages multiple language servers with robust error handling."""

    def __init__(self, root: Path):
        self.root = root
        self.clients: Dict[str, LSPClient] = {}
        self.disabled_languages: set[str] = set()
        self.failure_counts: Dict[str, int] = {}
        self.max_failures = 3  # Disable LSP after repeated failures

    # ------------------------------------------------------------------
    # Resolve executable paths
    # ------------------------------------------------------------------
    def _resolve_command(self, command: List[str]) -> Optional[List[str]]:
        exe = shutil.which(command[0])
        if exe:
            return [exe, *command[1:]]

        # Fallback for Pyright installed via uv/pip
        if command[0] == "pyright-langserver":
            return [sys.executable, "-m", "pyright.langserver", *command[1:]]

        print(f"[LSP WARNING] Executable not found: {command[0]}")
        return None

    # ------------------------------------------------------------------
    # Define language server commands
    # ------------------------------------------------------------------
    def _get_command(self, language: str) -> Optional[List[str]]:
        commands = {
            "python": ["pyright-langserver", "--stdio"],
            "typescript": ["typescript-language-server", "--stdio"],
            "tsx": ["typescript-language-server", "--stdio"],
            "javascript": ["typescript-language-server", "--stdio"],
            "jsx": ["typescript-language-server", "--stdio"],
        }
    
        # Special handling for PHP (Intelephense)
        if language == "php":
            node = shutil.which("node")
            intelephense = shutil.which("intelephense")
    
            if not node or not intelephense:
                print("[LSP WARNING] Node.js or Intelephense not found.")
                return None
    
            # Resolve the actual JS entry file
            intelephense_path = Path(intelephense).resolve()
            js_entry = (
                intelephense_path.parent.parent
                / "lib"
                / "node_modules"
                / "intelephense"
                / "lib"
                / "intelephense.js"
            )
    
            if not js_entry.exists():
                # Fallback: use the wrapper
                return [intelephense, "--stdio"]
    
            return [
                node,
                "--max-old-space-size=4096",
                str(js_entry),
                "--stdio",
            ]
    
        command = commands.get(language)
        return self._resolve_command(command) if command else None

    # ------------------------------------------------------------------
    # Retrieve or create LSP client
    # ------------------------------------------------------------------
    def get_client(self, language: str) -> Optional[LSPClient]:
        if language in self.disabled_languages:
            return None

        if language not in self.clients:
            command = self._get_command(language)
            if not command:
                self.disabled_languages.add(language)
                return None

            try:
                self.clients[language] = LSPClient(command, self.root)
            except Exception as exc:
                print(f"[LSP ERROR] Failed to start {language} server: {exc}")
                self.disabled_languages.add(language)
                return None

        return self.clients.get(language)

    # ------------------------------------------------------------------
    # Get definition with automatic recovery
    # ------------------------------------------------------------------
    def get_definition(
        self,
        file_path: Path,
        language: str,
        line: int,
        character: int,
    ):
        if language in self.disabled_languages:
            return None

        try:
            client = self.get_client(language)
            if not client:
                return None
            return client.get_definition(file_path, line, character)

        except BrokenPipeError:
            print(f"[LSP ERROR] {language}: Broken pipe.")
            self._restart_client(language)

        except Exception as exc:
            print(f"[LSP ERROR] {language}: {exc}")
            self._restart_client(language)

        return None

    # ------------------------------------------------------------------
    # Restart language server after failure
    # ------------------------------------------------------------------
    def _restart_client(self, language: str) -> None:
        self.failure_counts[language] = self.failure_counts.get(language, 0) + 1

        # Remove old client
        client = self.clients.pop(language, None)
        if client:
            try:
                client.shutdown()
            except Exception:
                pass

        # Disable language if failures exceed threshold
        if self.failure_counts[language] >= self.max_failures:
            print(f"[LSP WARNING] Disabling {language} LSP after repeated failures.")
            self.disabled_languages.add(language)

    # ------------------------------------------------------------------
    # Shutdown all language servers
    # ------------------------------------------------------------------
    def shutdown(self):
        for client in self.clients.values():
            try:
                client.shutdown()
            except Exception:
                pass

        self.clients.clear()
        self.failure_counts.clear()
        self.disabled_languages.clear()

In [14]:
import re
from typing import List, Tuple


class MermaidGraphBuilder:
    """Generates valid Mermaid diagrams for code relationships."""

    @staticmethod
    def _sanitize_id(text: str) -> str:
        """Create a Mermaid-safe node ID."""
        if not text:
            return "node"
        # Replace non-alphanumeric characters with underscores
        sanitized = re.sub(r"[^a-zA-Z0-9]", "_", text)
        # Ensure the ID does not start with a digit
        if sanitized[0].isdigit():
            sanitized = f"node_{sanitized}"
        return sanitized.strip("_") or "node"

    @staticmethod
    def _escape_label(text: str) -> str:
        """Escape quotes inside labels."""
        return text.replace('"', '\\"')

    def build_flowchart(self, relations: List[Tuple[str, str]]) -> str:
        lines = ["```mermaid", "graph TD"]
        declared_nodes = set()

        if not relations:
            lines.append('    NoData["No Data"] --> NoDefinitions["No Definitions Found"]')
        else:
            for source, target in relations:
                src_id = self._sanitize_id(source)
                tgt_id = self._sanitize_id(target)

                src_label = self._escape_label(source)
                tgt_label = self._escape_label(target)

                if src_id not in declared_nodes:
                    lines.append(f'    {src_id}["{src_label}"]')
                    declared_nodes.add(src_id)

                if tgt_id not in declared_nodes:
                    lines.append(f'    {tgt_id}["{tgt_label}"]')
                    declared_nodes.add(tgt_id)

                lines.append(f"    {src_id} --> {tgt_id}")

        lines.append("```")
        return "\n".join(lines)

In [48]:
import re
from urllib.parse import urlparse, unquote
from pathlib import Path
from typing import Callable, Iterable, Optional, Tuple, Dict, List


class ContextEngine:
    def __init__(
        self,
        config: EngineConfig | None = None,
        app_config: AppConfig | None = None,
        config_path: Path | None = None,
    ) -> None:
        self.app_config = app_config or load_app_config(config_path)
        self.config = config or self.app_config.engine
        self.indexed_root: Path | None = None
        self.files = []
        self.chunks = []
        self.graph = DependencyGraph()
        self.builder = ContextBuilder()
        self.llm_provider = create_llm_provider(self.app_config.llm)
        self.retriever: HybridRetriever | None = None
        self.embeddings_enabled = False
        self.lsp_manager: LSPManager | None = None
        self.mermaid_builder = MermaidGraphBuilder()
        self.symbol_index = self._build_symbol_index()

        # Cache to prevent repeated LSP calls
        self._definition_cache: Dict[Tuple[str, str], Optional[str]] = {}

    # ------------------------------------------------------------------
    # Configuration Management
    # ------------------------------------------------------------------
    def reload_config(self, config_path: Path | None = None) -> None:
        self.app_config = load_app_config(config_path)
        self.config = self.app_config.engine
        self.llm_provider = create_llm_provider(self.app_config.llm)
        if self.indexed_root:
            self.index_codebase(
                self.indexed_root,
                use_embeddings=self.embeddings_enabled,
            )

    # ------------------------------------------------------------------
    # Indexing
    # ------------------------------------------------------------------
    def index_codebase(
        self,
        root: Path,
        use_embeddings: bool = False,
        progress_callback: Callable[[int, int, Path], None] | None = None,
    ) -> None:
        root = root.resolve()
        indexer = CodebaseIndexer(self.config)
        files, chunks = indexer.index(
            root, progress_callback=progress_callback
        )
        self._set_index(root, files, chunks, use_embeddings)

    def index_uploaded_codebase(
        self,
        virtual_root: Path,
        uploaded_files: Iterable[tuple[Path, str]],
        use_embeddings: bool = False,
        progress_callback: Callable[[int, int, Path], None] | None = None,
    ) -> None:
        indexer = CodebaseIndexer(self.config)
        files, chunks = indexer.index_uploaded(
            virtual_root,
            uploaded_files,
            progress_callback=progress_callback,
        )
        self._set_index(virtual_root, files, chunks, use_embeddings)

    # def _set_index(
    #     self,
    #     root: Path,
    #     files,
    #     chunks,
    #     use_embeddings: bool = False,
    # ) -> None:
    #     self.files = files
    #     self.chunks = chunks
    #     self.graph.build(self.files)

    #     embedding_provider = (
    #         create_embedding_provider(self.app_config.embeddings)
    #         if use_embeddings
    #         else None
    #     )

    #     self.retriever = HybridRetriever(
    #         self.files,
    #         self.chunks,
    #         self.graph,
    #         embedding_provider=embedding_provider,
    #     )

    #     self.indexed_root = root
    #     self.lsp_manager = LSPManager(root)
    #     self.embeddings_enabled = use_embeddings

    #     # Clear cache on reindex
    #     self._definition_cache.clear()

    def _set_index(self, root: Path, files, chunks, use_embeddings: bool = False) -> None:
        self.files = files
        self.chunks = chunks
        self.graph.build(self.files)
    
        embedding_provider = (
            create_embedding_provider(self.app_config.embeddings)
            if use_embeddings
            else None
        )
    
        self.retriever = HybridRetriever(
            self.files,
            self.chunks,
            self.graph,
            embedding_provider=embedding_provider,
        )
    
        self.indexed_root = root
        self.lsp_manager = LSPManager(root)
    
        # ✅ Build global symbol index
        self.symbol_index = self._build_symbol_index()
    
        self.embeddings_enabled = use_embeddings

    # ------------------------------------------------------------------
    # Query & Chat
    # ------------------------------------------------------------------
    def query(self, query: str, top_k: int | None = None) -> QueryResponse:
        if not self.retriever:
            raise RuntimeError("Codebase is not indexed.")

        limit = top_k or self.config.top_k_chunks
        ranked = self.retriever.search(query, top_k=limit)
        context_pack = self.builder.build(query, ranked)

        return QueryResponse(
            query=query,
            indexed_root=self.indexed_root,
            total_files=len(self.files),
            total_chunks=len(self.chunks),
            context_pack=context_pack,
            ranked_chunks=ranked,
        )

    def chat(self, message: str, top_k: int | None = None) -> ChatResponse:
        response = self.query(message, top_k=top_k)

        system_prompt = (
            "You are an expert coding assistant. Use the provided repository "
            "context and be explicit when the context is insufficient."
        )

        answer = self.llm_provider.chat(
            [
                {"role": "system", "content": system_prompt},
                {
                    "role": "user",
                    "content": (
                        f"{response.context_pack}\n\nUser Message:\n{message}"
                    ),
                },
            ],
            temperature=self.app_config.llm.temperature,
        )

        return ChatResponse(
            message=message,
            indexed_root=self.indexed_root,
            model_provider=self.app_config.llm.provider,
            model_name=self.app_config.llm.model,
            context_pack=response.context_pack,
            answer=answer,
        )

    import re

    def _extract_called_symbols(self, content: str) -> set[str]:
        """Extract function and class calls from source code."""
        pattern = re.compile(r"\b([A-Za-z_][A-Za-z0-9_]*)\s*\(")
        return set(pattern.findall(content))

    def _build_symbol_index(self) -> dict[str, Path]:
        """Create a mapping from symbol names to their defining file paths."""
        symbol_index: dict[str, Path] = {}
    
        for file in self.files:
            for symbol in file.symbols:
                symbol_index[symbol.name] = file.path
    
        return symbol_index
    # ------------------------------------------------------------------
    # Definition Graph Generation
    # ------------------------------------------------------------------
    # def generate_definition_graph(
    #     self, query: str, top_k: int = 5
    # ) -> str:
    #     response = self.query(query, top_k=top_k)
    #     relations = []
    #     seen_relations = set()

    #     if not self.lsp_manager:
    #         return self.mermaid_builder.build_flowchart([])

    #     for ranked in response.ranked_chunks:
    #         chunk = ranked.chunk
    #         language = detect_language(chunk.file_path)

    #         if not chunk.symbols:
    #             continue

    #         for symbol in chunk.symbols:
    #             cache_key = (str(chunk.file_path), symbol)

    #             if cache_key in self._definition_cache:
    #                 target = self._definition_cache[cache_key]
    #             else:
    #                 target = None

    #                 # Step 1: Locate usage for LSP
    #                 line, col = self._find_symbol_position(
    #                     chunk.file_path, symbol
    #                 )

    #                 if line is not None:
    #                     try:
    #                         result = self.lsp_manager.get_definition(
    #                             chunk.file_path,
    #                             language,
    #                             line,
    #                             col,
    #                         )
    #                         target = self._extract_definition_name(result)
    #                     except Exception:
    #                         target = None

    #                 # Step 2: Fallback to local definition
    #                 if not target:
    #                     target = self._find_local_definition(
    #                         chunk.file_path, symbol
    #                     )

    #                 self._definition_cache[cache_key] = target

    #             if target:
    #                 relation = (symbol, target)
    #                 if relation not in seen_relations:
    #                     relations.append(relation)
    #                     seen_relations.add(relation)

    #     return self.mermaid_builder.build_flowchart(relations)

    def generate_definition_graph(
        self,
        query: str,
        top_k: int = 5,
        max_depth: int = 3,
        max_edges: int = 60,
        include_tests: bool = False,
    ) -> str:
        """Generate a readable recursive dependency graph."""
        response = self.query(query, top_k=top_k)
        relations: set[tuple[str, str]] = set()
    
        if not self.lsp_manager:
            return "LSP is not initialized."
    
        symbol_index = getattr(self, "symbol_index", {})
        visited: set[str] = set()
    
        def is_relevant_file(path: Path) -> bool:
            """Filter out noisy files like tests and configs."""
            parts = {p.lower() for p in path.parts}
    
            if not include_tests and any(
                p.startswith("test") or p == "tests" for p in parts
            ):
                return False
    
            excluded = {
                "site-packages",
                ".git",
                ".ipynb_checkpoints",
                ".venv",
                "venv",
                "node_modules",
                "__pycache__",
                "dist",
                "build"
            }
    
            return not any(p in excluded for p in parts)
    
        def traverse(symbol: str, depth: int):
            if (
                depth > max_depth
                or symbol in visited
                or len(relations) >= max_edges
            ):
                return
    
            visited.add(symbol)
            file_path = symbol_index.get(symbol)
            if not file_path or not is_relevant_file(file_path):
                return
    
            try:
                content = file_path.read_text(
                    encoding="utf-8", errors="ignore"
                )
            except Exception:
                return
    
            called_symbols = self._extract_called_symbols(content)
    
            for called in called_symbols:
                if called in symbol_index and called != symbol:
                    relations.add((symbol, called))
                    traverse(called, depth + 1)
    
        # Seed traversal from retrieved chunks
        for ranked in response.ranked_chunks:
            for symbol in ranked.chunk.symbols:
                traverse(symbol, 0)
                relations.add((symbol, ranked.chunk.file_path.stem))
    
        # Remove self-loops
        relations = {(s, t) for s, t in relations if s != t}
    
        # Limit output size
        relations = set(list(relations)[:max_edges])
    
        return self.mermaid_builder.build_flowchart(sorted(relations))

    def generate_file_dependency_graph(
        self,
        query: str | None = None,
        top_k: int = 10,
        include_tests: bool = False,
    ) -> str:
        """Generate a Mermaid graph showing file-level dependencies."""
        import re
        from pathlib import Path
    
        relations: set[tuple[str, str]] = set()
    
        if not self.files:
            return self.mermaid_builder.build_flowchart([])
    
        # Determine files to include
        selected_files: set[Path]
        if query:
            response = self.query(query, top_k=top_k)
            selected_files = {
                ranked.chunk.file_path for ranked in response.ranked_chunks
            }
        else:
            selected_files = {file.path for file in self.files}
    
        def is_relevant(path: Path) -> bool:
            parts = {p.lower() for p in path.parts}
    
            if not include_tests and any(
                p.startswith("test") or p == "tests" for p in parts
            ):
                return False
    
            excluded = {
                "__pycache__",
                ".venv",
                "node_modules",
                "site-packages",
            }
    
            return not any(p in excluded for p in parts)
    
        # Map module names to file paths
        module_map: dict[str, Path] = {}
        for file in self.files:
            if not self.indexed_root:
                continue
            rel = file.path.relative_to(self.indexed_root)
            module_name = ".".join(rel.with_suffix("").parts)
            module_map[module_name] = file.path
            module_map[file.path.stem] = file.path  # fallback
    
        def resolve_import(import_stmt: str, current_file: Path) -> Path | None:
            """Resolve an import statement to a project file."""
            import_stmt = import_stmt.strip()
    
            # Match Python imports
            match = re.match(
                r"(?:from\s+([\.\w]+)\s+import\s+[\w\*, ]+)|(?:import\s+([\.\w]+))",
                import_stmt,
            )
            if not match:
                return None
    
            module = match.group(1) or match.group(2)
            if not module:
                return None
    
            # Handle relative imports
            if module.startswith("."):
                level = len(module) - len(module.lstrip("."))
                base = current_file.parent
    
                for _ in range(level - 1):
                    base = base.parent
    
                module = module.lstrip(".")
                if module:
                    target = base / (module.replace(".", "/") + ".py")
                else:
                    target = base / "__init__.py"
    
                if target.exists():
                    return target.resolve()
                return None
    
            # Handle absolute imports
            if module in module_map:
                return module_map[module]
    
            # Try progressive resolution
            parts = module.split(".")
            for i in range(len(parts), 0, -1):
                candidate = ".".join(parts[:i])
                if candidate in module_map:
                    return module_map[candidate]
    
            return None
    
        # Build relationships
        for file in self.files:
            if file.path not in selected_files:
                continue
            if not is_relevant(file.path):
                continue
    
            source = str(file.path.relative_to(self.indexed_root))
    
            for imp in file.imports:
                target_path = resolve_import(imp, file.path)
                if not target_path:
                    continue
                if not is_relevant(target_path):
                    continue
    
                target = str(target_path.relative_to(self.indexed_root))
                relations.add((source, target))
    
        return self.mermaid_builder.build_flowchart(sorted(relations))

    # ------------------------------------------------------------------
    # Extract Definition Name from LSP Result
    # ------------------------------------------------------------------
    def _extract_definition_name(self, result):
        if isinstance(result, list) and result:
            result = result[0]

        if not isinstance(result, dict):
            return None

        uri = result.get("uri")
        if not uri:
            return None

        parsed = urlparse(uri)
        file_path = unquote(parsed.path)

        # Fix Windows paths
        if file_path.startswith("/") and ":" in file_path:
            file_path = file_path[1:]

        return Path(file_path).stem

    # ------------------------------------------------------------------
    # STEP 1: Locate Symbol Usage for LSP
    # ------------------------------------------------------------------
    def _find_symbol_position(
        self, file_path: Path, symbol: str
    ) -> Tuple[Optional[int], Optional[int]]:
        try:
            pattern = re.compile(rf"\b{re.escape(symbol)}\b")
            lines = file_path.read_text(
                encoding="utf-8", errors="ignore"
            ).splitlines()

            for line_no, line in enumerate(lines, start=1):
                stripped = line.strip()

                # Skip obvious definitions and imports
                if stripped.startswith(
                    (
                        "def ",
                        "class ",
                        "import ",
                        "from ",
                        "function ",
                        "public function",
                        "private function",
                        "protected function",
                    )
                ):
                    continue

                match = pattern.search(line)
                if match:
                    return line_no, match.start()

        except Exception:
            pass

        return None, None

    # ------------------------------------------------------------------
    # STEP 2: Fallback for Local Definitions
    # ------------------------------------------------------------------
    def _find_local_definition(
        self, file_path: Path, symbol: str
    ) -> Optional[str]:
        try:
            content = file_path.read_text(
                encoding="utf-8", errors="ignore"
            )

            patterns = [
                rf"class\s+{re.escape(symbol)}\b",
                rf"function\s+{re.escape(symbol)}\b",
                rf"public\s+function\s+{re.escape(symbol)}\b",
                rf"protected\s+function\s+{re.escape(symbol)}\b",
                rf"private\s+function\s+{re.escape(symbol)}\b",
                rf"def\s+{re.escape(symbol)}\b",  # Python fallback
            ]

            for pattern in patterns:
                if re.search(pattern, content):
                    return file_path.stem

        except Exception:
            pass

        return None

    # ------------------------------------------------------------------
    # Cleanup
    # ------------------------------------------------------------------
    def shutdown(self) -> None:
        if self.lsp_manager:
            self.lsp_manager.shutdown()

In [60]:
engine = ContextEngine(config_path=Path("/home/butcher/MyCodeIDE/config.toml"))

In [61]:
engine.index_codebase(root=Path(r"/home/butcher/Downloads/cve_wesbite_wordpress_110"))

In [64]:
resp = engine.query(query="function that catches form?", top_k=4)

In [65]:
resp.query

'function that catches form?'

In [66]:
print("query: ", resp.query)
print("root: ", resp.indexed_root)
print("Total Files: ", resp.total_files)
print("Total Chunks: ", resp.total_chunks)

query:  function that catches form?
root:  /home/butcher/Downloads/cve_wesbite_wordpress_110
Total Files:  260
Total Chunks:  1847


In [67]:
#print(resp.context_pack)

In [68]:
for idx, r in enumerate(resp.ranked_chunks):
    chunk = r.chunk

    print(f"<==============Rank: {idx}===============>\n")
    print("file_path: ", chunk.file_path)
    print("start_line: ", chunk.start_line)
    print("end_line: ", chunk.end_line)
    print("reason: ", r.reasons)
    print("score: ", r.score)
    print("symbols: ", chunk.symbols)
    print("content: \n", chunk.content)
    print("\n<=======================================================================================================================================>\n\n")

<==============Rank: 0===============>

file_path:  /home/butcher/Downloads/cve_wesbite_wordpress_110/cve_wesbite_wordpress/wp-content/plugins/akismet/class.akismet.php
start_line:  1765
end_line:  1792
reason:  ['bm25 keyword match', 'fuzzy match', 'symbol/path match']
score:  3.7103
symbols:  ['get_akismet_form_fields']
content: 
 public static function get_akismet_form_fields() {
		$fields = '';

		$prefix = 'ak_';

		// Contact Form 7 uses _wpcf7 as a prefix to know which fields to exclude from comment_content.
		if ( 'wpcf7_form_elements' === current_filter() ) {
			$prefix = '_wpcf7_ak_';
		}

		$fields .= '<p style="display: none !important;" class="akismet-fields-container" data-prefix="' . esc_attr( $prefix ) . '">';
		$fields .= '<label>&#916;<textarea name="' . $prefix . 'hp_textarea" cols="45" rows="8" maxlength="100"></textarea></label>';

		if ( ! function_exists( 'amp_is_request' ) || ! amp_is_request() ) {
			// Keep track of how many ak_js fields are in this page so th

In [69]:
from IPython.display import Markdown, display

mermaid_graph = engine.generate_definition_graph(
    "function that catches form?",
    top_k=20,
    max_depth=3,
    max_edges=60
)

display(Markdown(mermaid_graph))

```mermaid
graph TD
    auto_check_comment["auto_check_comment"]
    cmp_time["_cmp_time"]
    auto_check_comment --> cmp_time
    get_microtime["_get_microtime"]
    auto_check_comment --> get_microtime
    allow_discard["allow_discard"]
    auto_check_comment --> allow_discard
    append_custom_form_fields["append_custom_form_fields"]
    auto_check_comment --> append_custom_form_fields
    deactivate_key["deactivate_key"]
    auto_check_comment --> deactivate_key
    get_ip_address["get_ip_address"]
    auto_check_comment --> get_ip_address
    init["init"]
    auto_check_comment --> init
    init_hooks["init_hooks"]
    auto_check_comment --> init_hooks
    log["log"]
    auto_check_comment --> log
    plugin_activation["plugin_activation"]
    auto_check_comment --> plugin_activation
    schedule_approval_fallback["schedule_approval_fallback"]
    auto_check_comment --> schedule_approval_fallback
    set_last_comment["set_last_comment"]
    auto_check_comment --> set_last_comment
    update_alert["update_alert"]
    auto_check_comment --> update_alert
    update_comment_history["update_comment_history"]
    auto_check_comment --> update_comment_history
    verify_key["verify_key"]
    auto_check_comment --> verify_key
    build_query["build_query"]
    add_comment_nonce["add_comment_nonce"]
    build_query --> add_comment_nonce
    added_option["added_option"]
    build_query --> added_option
    akismet_result_spam["akismet_result_spam"]
    build_query --> akismet_result_spam
    build_query --> auto_check_comment
    bail_on_activation["bail_on_activation"]
    build_query --> bail_on_activation
    check_db_comment["check_db_comment"]
    build_query --> check_db_comment
    check_server_connectivity["check_server_connectivity"]
    build_query --> check_server_connectivity
    get_fields_for_comment_matching["get_fields_for_comment_matching"]
    build_query --> get_fields_for_comment_matching
    get_last_comment["get_last_comment"]
    build_query --> get_last_comment
    get_referer["get_referer"]
    build_query --> get_referer
    http_post["http_post"]
    build_query --> http_post
    plugin_deactivation["plugin_deactivation"]
    build_query --> plugin_deactivation
    sanitize_comment_as_submitted["sanitize_comment_as_submitted"]
    build_query --> sanitize_comment_as_submitted
    admin_head["admin_head"]
    check_server_connectivity --> admin_head
    admin_init["admin_init"]
    check_server_connectivity --> admin_init
    admin_menu["admin_menu"]
    check_server_connectivity --> admin_menu
    check_server_connectivity --> build_query
    display_stats_page["display_stats_page"]
    check_server_connectivity --> display_stats_page
    display_status["display_status"]
    check_server_connectivity --> display_status
    get_akismet_user["get_akismet_user"]
    check_server_connectivity --> get_akismet_user
    get_jetpack_user["get_jetpack_user"]
    check_server_connectivity --> get_jetpack_user
    get_usage_limit_alert_data["get_usage_limit_alert_data"]
    check_server_connectivity --> get_usage_limit_alert_data
    check_server_connectivity --> init
    check_server_connectivity --> init_hooks
    load_menu["load_menu"]
    check_server_connectivity --> load_menu
    load_resources["load_resources"]
    check_server_connectivity --> load_resources
    check_server_connectivity --> log
    plugin_action_links["plugin_action_links"]
    check_server_connectivity --> plugin_action_links
    recheck_comment["recheck_comment"]
    check_server_connectivity --> recheck_comment
    register_personal_data_eraser["register_personal_data_eraser"]
    check_server_connectivity --> register_personal_data_eraser
    rightnow_stats["rightnow_stats"]
    check_server_connectivity --> rightnow_stats
    check_server_connectivity --> verify_key
    get_akismet_form_fields["get_akismet_form_fields"]
    get_akismet_form_fields --> get_microtime
    get_akismet_form_fields --> allow_discard
    get_akismet_form_fields --> deactivate_key
    delete_old_comments_meta["delete_old_comments_meta"]
    get_akismet_form_fields --> delete_old_comments_meta
    get_akismet_form_fields --> get_ip_address
    get_akismet_form_fields --> init_hooks
    matches_last_comment["matches_last_comment"]
    get_akismet_form_fields --> matches_last_comment
    get_akismet_form_fields --> schedule_approval_fallback
    get_akismet_form_fields --> set_last_comment
    submit_spam_comment["submit_spam_comment"]
    get_akismet_form_fields --> submit_spam_comment
    updated_option["updated_option"]
    get_akismet_form_fields --> updated_option
    get_akismet_form_fields --> verify_key
    view["view"]
    get_akismet_form_fields --> view
```

In [70]:
graph = engine.generate_file_dependency_graph()
display(Markdown(graph))

```mermaid
graph TD
    NoData["No Data"] --> NoDefinitions["No Definitions Found"]
```

In [71]:
graph = engine.generate_file_dependency_graph(
    query="function that catches form?",
    top_k=10
)
display(Markdown(graph))

```mermaid
graph TD
    NoData["No Data"] --> NoDefinitions["No Definitions Found"]
```

In [47]:
print(graph)

```mermaid
graph TD
    NoData["No Data"] --> NoDefinitions["No Definitions Found"]
```


In [33]:
from IPython.display import Markdown, display

mermaid_graph = engine.generate_definition_graph(
    "how browser mcp tool works?",
    top_k=15
)

display(Markdown(mermaid_graph))

```mermaid
graph TD
    filterButcherTools["filterButcherTools"]
    ai_tools["ai-tools"]
    filterButcherTools --> ai_tools
    isConnectionEnabled["isConnectionEnabled"]
    isConnectionEnabled --> ai_tools
    node[") => "]
    ToolDetailIoViews["ToolDetailIoViews"]
    node --> ToolDetailIoViews
    isToolEnabled["isToolEnabled"]
    isToolEnabled --> ai_tools
    systemPromptForWorkspace["systemPromptForWorkspace"]
    systemPromptForWorkspace --> ai_tools
    executeExternalTool["executeExternalTool"]
    executeExternalTool --> ai_tools
```

In [34]:
from IPython.display import Markdown, display

mermaid_graph = engine.generate_definition_graph(
    "how browser agent mcp tool works?",
    top_k=15
)

display(Markdown(mermaid_graph))

```mermaid
graph TD
    node[") => "]
    ToolDetailIoViews["ToolDetailIoViews"]
    node --> ToolDetailIoViews
    systemPromptForWorkspace["systemPromptForWorkspace"]
    ai_system_prompts["ai-system-prompts"]
    systemPromptForWorkspace --> ai_system_prompts
    filterButcherTools["filterButcherTools"]
    ai_tools["ai-tools"]
    filterButcherTools --> ai_tools
    isConnectionEnabled["isConnectionEnabled"]
    isConnectionEnabled --> ai_system_prompts
```